# for hscd lec slides

In [2]:
from pathlib import Path
import re
from collections import defaultdict

INPUT_DIR = Path(r"E:\OneDrive - MSFT\.master_data\25-26ws\hscd\2311620 n Hardware-Software Co-Design (WS 25_26)\Lecture slides")
OUTPUT_DIR = INPUT_DIR / "_merged"
OUTPUT_DIR.mkdir(exist_ok=True)

EXTS = {".ppt", ".pptx", ".pdf"}

def numeric_key(path):
    return tuple(int(x) for x in re.findall(r"\d+", path.stem))

def get_hsc_group(path):
    """
    HSC_2_1_Target_Architectures_Structure.pdf -> HSC_2
    HSC_2_3_1_Target_Architectures_Pipelining.pdf -> HSC_2
    HSC_3_1_xxx.pptx -> HSC_3
    """
    m = re.match(r"^HSC_(\d+)_", path.stem, re.IGNORECASE)
    if not m:
        return None
    return f"HSC_{int(m.group(1))}"

def count_pdf_pages(path):
    try:
        from pypdf import PdfReader
    except ImportError:
        raise ImportError("请先运行：pip install pypdf")

    return len(PdfReader(str(path)).pages)

def merge_pdfs(files, output_path):
    try:
        from pypdf import PdfWriter
    except ImportError:
        raise ImportError("请先运行：pip install pypdf")

    writer = PdfWriter()
    file_pages = []

    for f in files:
        n = count_pdf_pages(f)
        file_pages.append((f, n))
        writer.append(str(f))

    with open(output_path, "wb") as out:
        writer.write(out)

    total_pages = count_pdf_pages(output_path)
    return file_pages, total_pages

def merge_ppts(files, output_path):
    try:
        import win32com.client as win32
    except ImportError:
        raise ImportError("请先运行：pip install pywin32。该方法需要 Windows + PowerPoint。")

    app = win32.Dispatch("PowerPoint.Application")
    app.Visible = True

    merged = app.Presentations.Add()
    file_pages = []

    try:
        while merged.Slides.Count > 0:
            merged.Slides(1).Delete()

        for f in files:
            src = app.Presentations.Open(str(f.resolve()), ReadOnly=True, WithWindow=False)
            n = src.Slides.Count
            file_pages.append((f, n))
            src.Close()

            if n > 0:
                merged.Slides.InsertFromFile(str(f.resolve()), merged.Slides.Count, 1, n)

        total_pages = merged.Slides.Count
        merged.SaveAs(str(output_path.resolve()))

    finally:
        merged.Close()
        app.Quit()

    return file_pages, total_pages

files = [
    p for p in INPUT_DIR.rglob("*")
    if p.is_file()
    and p.suffix.lower() in EXTS
    and not p.name.startswith("~$")
    and "_merged" not in p.parts
]

groups = defaultdict(list)

for f in files:
    group = get_hsc_group(f)
    if group:
        groups[group].append(f)

for group, fs in sorted(groups.items(), key=lambda x: numeric_key(Path(x[0]))):
    fs = sorted(fs, key=numeric_key)

    print("\n" + "=" * 80)
    print(f"分组：{group}")
    print(f"合并前文件数：{len(fs)}")

    ppt_files = [f for f in fs if f.suffix.lower() in {".ppt", ".pptx"}]
    pdf_files = [f for f in fs if f.suffix.lower() == ".pdf"]

    if ppt_files:
        out = OUTPUT_DIR / f"{group}.pptx"
        file_pages, total_pages = merge_ppts(ppt_files, out)

        print("\nPPT 文件：")
        for f, n in file_pages:
            print(f"  {f.name} -> {n} 页")
        print(f"合并后文件：{out}")
        print(f"合并后总页数：{total_pages}")

    if pdf_files:
        out = OUTPUT_DIR / f"{group}.pdf"
        file_pages, total_pages = merge_pdfs(pdf_files, out)

        print("\nPDF 文件：")
        for f, n in file_pages:
            print(f"  {f.name} -> {n} 页")
        print(f"合并后文件：{out}")
        print(f"合并后总页数：{total_pages}")


分组：HSC_0
合并前文件数：1

PDF 文件：
  HSC_0_Infomation.pdf -> 27 页
合并后文件：E:\OneDrive - MSFT\.master_data\25-26ws\hscd\2311620 n Hardware-Software Co-Design (WS 25_26)\Lecture slides\_merged\HSC_0.pdf
合并后总页数：27

分组：HSC_1
合并前文件数：1

PDF 文件：
  HSC_1_1_introduction.pdf -> 36 页
合并后文件：E:\OneDrive - MSFT\.master_data\25-26ws\hscd\2311620 n Hardware-Software Co-Design (WS 25_26)\Lecture slides\_merged\HSC_1.pdf
合并后总页数：36

分组：HSC_2
合并前文件数：17

PDF 文件：
  HSC_2_1_Target_Architectures_Structure.pdf -> 26 页
  HSC_2_2_Target_Architectures_Classification.pdf -> 21 页
  HSC_2_3_1_Target_Architectures_Pipelining.pdf -> 57 页
  HSC_2_3_2_Target_Architectures_Superscalarity.pdf -> 44 页
  HSC_2_3_3_Target_Architectures_VLIW.pdf -> 20 页
  HSC_2_3_4_Target_Architectures_SIMD.pdf -> 16 页
  HSC_2_3_5_Target_Architectures_Cache.pdf -> 29 页
  HSC_2_3_6_Target_Architectures_MIMD.pdf -> 25 页
  HSC_2_4_Target_Architectures_Microcontroller.pdf -> 48 页
  HSC_2_5_Target_Architectures_DSP.pdf -> 22 页
  HSC_2_6_Target_Architecture

拆分大页码，为200

In [1]:
from pathlib import Path

try:
    from pypdf import PdfReader, PdfWriter
except ImportError:
    raise ImportError("请先运行：pip install pypdf")

PDF_PATH = Path(r"E:\OneDrive - MSFT\.master_data\25-26ws\hscd\2311620 n Hardware-Software Co-Design (WS 25_26)\Lecture slides\_merged\HSC_2.pdf")
MAX_PAGES = 200

reader = PdfReader(str(PDF_PATH))
total_pages = len(reader.pages)

output_dir = PDF_PATH.parent / f"{PDF_PATH.stem}_split_max_{MAX_PAGES}"
output_dir.mkdir(exist_ok=True)

print("=" * 80)
print(f"输入文件：{PDF_PATH}")
print(f"原始总页数：{total_pages}")
print(f"每个文件最多页数：{MAX_PAGES}")
print("=" * 80)

part_files = []

for start in range(0, total_pages, MAX_PAGES):
    end = min(start + MAX_PAGES, total_pages)
    part_id = start // MAX_PAGES + 1

    writer = PdfWriter()

    for page_idx in range(start, end):
        writer.add_page(reader.pages[page_idx])

    out_path = output_dir / f"{PDF_PATH.stem}_part{part_id:02d}_p{start + 1:03d}-{end:03d}.pdf"

    with open(out_path, "wb") as f:
        writer.write(f)

    part_pages = end - start
    part_files.append((out_path, part_pages))

    print(f"第 {part_id} 部分：第 {start + 1} 页到第 {end} 页，共 {part_pages} 页")
    print(f"输出文件：{out_path}")

print("=" * 80)
print(f"拆分后文件数：{len(part_files)}")
print(f"拆分后总页数：{sum(n for _, n in part_files)}")

if sum(n for _, n in part_files) == total_pages:
    print("检查通过：拆分前后总页数一致")
else:
    print("警告：拆分前后总页数不一致")

输入文件：E:\OneDrive - MSFT\.master_data\25-26ws\hscd\2311620 n Hardware-Software Co-Design (WS 25_26)\Lecture slides\_merged\HSC_2.pdf
原始总页数：547
每个文件最多页数：200
第 1 部分：第 1 页到第 200 页，共 200 页
输出文件：E:\OneDrive - MSFT\.master_data\25-26ws\hscd\2311620 n Hardware-Software Co-Design (WS 25_26)\Lecture slides\_merged\HSC_2_split_max_200\HSC_2_part01_p001-200.pdf
第 2 部分：第 201 页到第 400 页，共 200 页
输出文件：E:\OneDrive - MSFT\.master_data\25-26ws\hscd\2311620 n Hardware-Software Co-Design (WS 25_26)\Lecture slides\_merged\HSC_2_split_max_200\HSC_2_part02_p201-400.pdf
第 3 部分：第 401 页到第 547 页，共 147 页
输出文件：E:\OneDrive - MSFT\.master_data\25-26ws\hscd\2311620 n Hardware-Software Co-Design (WS 25_26)\Lecture slides\_merged\HSC_2_split_max_200\HSC_2_part03_p401-547.pdf
拆分后文件数：3
拆分后总页数：547
检查通过：拆分前后总页数一致


# for hscd exercise

In [3]:
from pathlib import Path
import re
from collections import defaultdict

INPUT_DIR = Path(r"E:\OneDrive - MSFT\.master_data\25-26ws\hscd\2311620 n Hardware-Software Co-Design (WS 25_26)\Exercise materials")
OUTPUT_DIR = INPUT_DIR / "_merged"
OUTPUT_DIR.mkdir(exist_ok=True)

EXTS = {".ppt", ".pptx", ".pdf"}

def get_ex_group(path):
    """
    EX01-tasks.pdf -> EX01
    EX01-tasks-solution.pdf -> EX01
    EX01-presentation.pdf -> EX01
    EX02-presentation.pptx -> EX02
    """
    m = re.match(r"^EX0*(\d+)", path.stem, re.IGNORECASE)
    if not m:
        return None
    return f"EX{int(m.group(1)):02d}"

def exercise_sort_key(path):
    name = path.stem.lower()

    if "presentation" in name:
        role = 0
    elif "tasks-solution" in name or "solution" in name:
        role = 2
    elif "tasks" in name:
        role = 1
    else:
        role = 9

    nums = tuple(int(x) for x in re.findall(r"\d+", name))
    return nums, role, name

def count_pdf_pages(path):
    try:
        from pypdf import PdfReader
    except ImportError:
        raise ImportError("请先运行：pip install pypdf")

    return len(PdfReader(str(path)).pages)

def merge_pdfs(files, output_path):
    try:
        from pypdf import PdfWriter
    except ImportError:
        raise ImportError("请先运行：pip install pypdf")

    writer = PdfWriter()
    file_pages = []

    for f in files:
        n = count_pdf_pages(f)
        file_pages.append((f, n))
        writer.append(str(f))

    with open(output_path, "wb") as out:
        writer.write(out)

    total_pages = count_pdf_pages(output_path)
    return file_pages, total_pages

def merge_ppts(files, output_path):
    try:
        import win32com.client as win32
    except ImportError:
        raise ImportError("请先运行：pip install pywin32。该方法需要 Windows + PowerPoint。")

    app = win32.Dispatch("PowerPoint.Application")
    app.Visible = True

    merged = app.Presentations.Add()
    file_pages = []

    try:
        while merged.Slides.Count > 0:
            merged.Slides(1).Delete()

        for f in files:
            src = app.Presentations.Open(str(f.resolve()), ReadOnly=True, WithWindow=False)
            n = src.Slides.Count
            file_pages.append((f, n))
            src.Close()

            if n > 0:
                merged.Slides.InsertFromFile(str(f.resolve()), merged.Slides.Count, 1, n)

        total_pages = merged.Slides.Count
        merged.SaveAs(str(output_path.resolve()))

    finally:
        merged.Close()
        app.Quit()

    return file_pages, total_pages

files = [
    p for p in INPUT_DIR.rglob("*")
    if p.is_file()
    and p.suffix.lower() in EXTS
    and not p.name.startswith("~$")
    and "_merged" not in p.parts
]

groups = defaultdict(list)

for f in files:
    group = get_ex_group(f)
    if group:
        groups[group].append(f)

for group, fs in sorted(groups.items()):
    fs = sorted(fs, key=exercise_sort_key)

    print("\n" + "=" * 80)
    print(f"分组：{group}")
    print(f"合并前文件数：{len(fs)}")

    ppt_files = [f for f in fs if f.suffix.lower() in {".ppt", ".pptx"}]
    pdf_files = [f for f in fs if f.suffix.lower() == ".pdf"]

    if ppt_files:
        out = OUTPUT_DIR / f"{group}.pptx"
        file_pages, total_pages = merge_ppts(ppt_files, out)

        print("\nPPT 文件：")
        for f, n in file_pages:
            print(f"  {f.name} -> {n} 页")
        print(f"合并后文件：{out}")
        print(f"合并后总页数：{total_pages}")

    if pdf_files:
        out = OUTPUT_DIR / f"{group}.pdf"
        file_pages, total_pages = merge_pdfs(pdf_files, out)

        print("\nPDF 文件：")
        for f, n in file_pages:
            print(f"  {f.name} -> {n} 页")
        print(f"合并后文件：{out}")
        print(f"合并后总页数：{total_pages}")


分组：EX01
合并前文件数：3

PDF 文件：
  EX01-presentation.pdf -> 43 页
  EX01-tasks.pdf -> 1 页
  EX01-tasks-solution.pdf -> 6 页
合并后文件：E:\OneDrive - MSFT\.master_data\25-26ws\hscd\2311620 n Hardware-Software Co-Design (WS 25_26)\Exercise materials\_merged\EX01.pdf
合并后总页数：50

分组：EX02
合并前文件数：3


Ignoring wrong pointing object 37 0 (offset 0)
Ignoring wrong pointing object 37 0 (offset 0)



PDF 文件：
  EX02-presentation.pdf -> 43 页
  EX02-tasks.pdf -> 2 页
  EX02-tasks-solutions.pdf -> 6 页
合并后文件：E:\OneDrive - MSFT\.master_data\25-26ws\hscd\2311620 n Hardware-Software Co-Design (WS 25_26)\Exercise materials\_merged\EX02.pdf
合并后总页数：51

分组：EX03
合并前文件数：4

PDF 文件：
  EX03-presentation.pdf -> 35 页
  EX03-tasks.pdf -> 3 页
  EX03-solution.pdf -> 4 页
  EX03-task3.3-handwritten-solution.pdf -> 1 页
合并后文件：E:\OneDrive - MSFT\.master_data\25-26ws\hscd\2311620 n Hardware-Software Co-Design (WS 25_26)\Exercise materials\_merged\EX03.pdf
合并后总页数：43

分组：EX04
合并前文件数：3

PDF 文件：
  EX04-presentation.pdf -> 46 页
  EX04-tasks.pdf -> 4 页
  EX04-solutions.pdf -> 4 页
合并后文件：E:\OneDrive - MSFT\.master_data\25-26ws\hscd\2311620 n Hardware-Software Co-Design (WS 25_26)\Exercise materials\_merged\EX04.pdf
合并后总页数：54

分组：EX05
合并前文件数：3

PDF 文件：
  EX05-presentation.pdf -> 34 页
  EX05-tasks.pdf -> 3 页
  EX05-solutions.pdf -> 4 页
合并后文件：E:\OneDrive - MSFT\.master_data\25-26ws\hscd\2311620 n Hardware-Software Co-De

裁剪不合适的翻译文件

In [4]:
from pathlib import Path
from pypdf import PdfReader, PdfWriter

# 只改这里
PDF_PATH = Path(r"E:\OneDrive - MSFT\.master_data\25-26ws\hscd\2311620 n Hardware-Software Co-Design (WS 25_26)\Lecture slides\_merged\_HSC_2_part03_p401-547.pdf")

# 裁掉多少，单位 mm
CROP_LEFT_MM = 0
CROP_TOP_MM = 0
CROP_RIGHT_MM = 51.5
CROP_BOTTOM_MM = 25

def mm_to_pt(mm):
    return mm * 72 / 25.4

reader = PdfReader(str(PDF_PATH))
writer = PdfWriter()

crop_left = mm_to_pt(CROP_LEFT_MM)
crop_top = mm_to_pt(CROP_TOP_MM)
crop_right = mm_to_pt(CROP_RIGHT_MM)
crop_bottom = mm_to_pt(CROP_BOTTOM_MM)

for page in reader.pages:
    page_width = float(page.mediabox.width)
    page_height = float(page.mediabox.height)

    # PDF 坐标原点在左下角
    new_left = crop_left
    new_bottom = crop_bottom
    new_right = page_width - crop_right
    new_top = page_height - crop_top

    page.cropbox.lower_left = (new_left, new_bottom)
    page.cropbox.upper_right = (new_right, new_top)

    writer.add_page(page)

out_path = PDF_PATH.with_name(PDF_PATH.stem + "_cropped.pdf")

with open(out_path, "wb") as f:
    writer.write(f)

print("=" * 80)
print(f"原始文件：{PDF_PATH}")
print(f"原始页数：{len(reader.pages)}")
print(f"裁掉左边：{CROP_LEFT_MM} mm")
print(f"裁掉上边：{CROP_TOP_MM} mm")
print(f"裁掉右边：{CROP_RIGHT_MM} mm")
print(f"裁掉下边：{CROP_BOTTOM_MM} mm")
print(f"输出文件：{out_path}")
print("=" * 80)

原始文件：E:\OneDrive - MSFT\.master_data\25-26ws\hscd\2311620 n Hardware-Software Co-Design (WS 25_26)\Lecture slides\_merged\_HSC_2_part03_p401-547.pdf
原始页数：294
裁掉左边：0 mm
裁掉上边：0 mm
裁掉右边：51.5 mm
裁掉下边：25 mm
输出文件：E:\OneDrive - MSFT\.master_data\25-26ws\hscd\2311620 n Hardware-Software Co-Design (WS 25_26)\Lecture slides\_merged\_HSC_2_part03_p401-547_cropped.pdf


翻译？


In [1]:
from pathlib import Path
import subprocess
import tempfile
import shutil

# 只改这里
FOLDER = Path(r"E:\OneDrive - MSFT\.master_data\25-26ws\hscd\2311620 n Hardware-Software Co-Design (WS 25_26)\Lecture slides\_merged\test")

# 翻译设置
SOURCE_LANG = "en"
TARGET_LANG = "zh"

# 默认用 pdf2zh 自带默认翻译服务
# 想指定服务可以写 "google", "deepl", "openai" 等
SERVICE = None

# 已经存在 -原文件名.pdf 时是否覆盖
OVERWRITE = False

pdf2zh_cmd = shutil.which("pdf2zh")
if pdf2zh_cmd is None:
    raise RuntimeError("没有找到 pdf2zh。请先运行：pip install pdf2zh")

pdf_files = sorted([
    p for p in FOLDER.glob("*.pdf")
    if p.is_file()
    and not p.name.startswith("-")
    and not p.name.startswith("~$")
])

print("=" * 80)
print(f"待翻译文件夹：{FOLDER}")
print(f"待翻译 PDF 数量：{len(pdf_files)}")
print("=" * 80)

for pdf_path in pdf_files:
    out_path = pdf_path.with_name("-" + pdf_path.name)

    if out_path.exists() and not OVERWRITE:
        print(f"跳过，已存在：{out_path.name}")
        continue

    print("\n" + "-" * 80)
    print(f"开始翻译：{pdf_path.name}")

    with tempfile.TemporaryDirectory() as tmpdir:
        tmpdir = Path(tmpdir)
        tmp_input = tmpdir / pdf_path.name
        shutil.copy2(pdf_path, tmp_input)

        cmd = [
            pdf2zh_cmd,
            str(tmp_input),
            "-li", SOURCE_LANG,
            "-lo", TARGET_LANG,
        ]

        if SERVICE:
            cmd += ["-s", SERVICE]

        result = subprocess.run(
            cmd,
            cwd=tmpdir,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT
        )

        print(result.stdout[-2000:])

        if result.returncode != 0:
            print(f"失败：{pdf_path.name}")
            continue

        # 优先找双语 dual 文件
        candidates = list(tmpdir.glob(f"{pdf_path.stem}*dual*.pdf"))

        # 兼容不同版本输出名
        if not candidates:
            candidates = list(tmpdir.glob(f"{pdf_path.stem}*zh*.pdf"))
        if not candidates:
            candidates = list(tmpdir.glob(f"{pdf_path.stem}*mono*.pdf"))

        if not candidates:
            print(f"没有找到翻译输出文件：{pdf_path.name}")
            continue

        translated_pdf = candidates[0]

        if out_path.exists() and OVERWRITE:
            out_path.unlink()

        shutil.move(str(translated_pdf), str(out_path))

        print(f"完成：{out_path.name}")

print("\n" + "=" * 80)
print("全部处理完成")
print("=" * 80)

待翻译文件夹：E:\OneDrive - MSFT\.master_data\25-26ws\hscd\2311620 n Hardware-Software Co-Design (WS 25_26)\Lecture slides\_merged\test
待翻译 PDF 数量：1

--------------------------------------------------------------------------------
开始翻译：HSC_0.pdf
45e'

MuPDF error: syntax error: unknown keyword: '6.010177727475438e'

MuPDF error: syntax error: unknown keyword: '1.5025444318688592e'

MuPDF error: syntax error: unknown keyword: '6.493589249874034e'

MuPDF error: syntax error: unknown keyword: '1.5574393910725945e'

MuPDF error: syntax error: unknown keyword: '6.010177727475438e'

MuPDF error: syntax error: unknown keyword: '1.5025444318688592e'

MuPDF error: syntax error: unknown keyword: '6.493589249874034e'

MuPDF error: syntax error: unknown keyword: '1.5574393910725945e'

MuPDF error: syntax error: unknown keyword: '1.143782647405874e'

MuPDF error: syntax error: unknown keyword: '3.652188734298674e'

MuPDF error: syntax error: unknown keyword: '6.010177727475438e'

MuPDF error: syntax error